# OpenPlaque — RCA Source-Space Plaque Excess Specificity v1

Reference-normalized source-space plaque specificity validation using the completed RCA source-space quantification as a prerequisite. The frozen master is never modified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, shutil
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT = DRIVE_ROOT / 'RCA_Source_Space_Plaque_Excess_Specificity_v1'
REUSE_VALID_OUTPUT = True
reuse_complete = False
if REUSE_VALID_OUTPUT and (OUTPUT / 'run_state.json').exists() and (OUTPUT / 'summary.json').exists():
    try:
        rs = json.loads((OUTPUT / 'run_state.json').read_text())
        reuse_complete = rs.get('status') == 'COMPLETE'
    except Exception:
        reuse_complete = False
if not reuse_complete and OUTPUT.exists():
    shutil.rmtree(OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'notebook_started.json').write_text(json.dumps({'status':'STARTED'}))
print('Output:', OUTPUT)
print('Reuse complete output:', reuse_complete)

In [ ]:
import os, sys
os.chdir('/content')
print('Working directory:', os.getcwd())
BRANCH = 'rca-source-space-plaque-excess-specificity-from-main'
PINNED_SCIENCE_COMMIT = 'ed89dd95de5b707a6624125978a9fe4722cd3261'
BASELINE = '0593b453959f5a353d644267fbeef24b514ef4d7'
REPO = Path('/content/OpenPlaque_rca_excess_v1')
if REPO.exists():
    shutil.rmtree(REPO)
get_ipython().system(f'git clone -q --branch {BRANCH} https://github.com/pazzani/OpenPlaque.git {REPO}')
get_ipython().system(f'git -C {REPO} checkout -q {PINNED_SCIENCE_COMMIT}')
HEAD = get_ipython().getoutput(f'git -C {REPO} rev-parse HEAD')[-1].strip()
MB = get_ipython().getoutput(f'git -C {REPO} merge-base HEAD origin/main')[-1].strip()
print('Checked out:', HEAD)
print('Merge base:', MB)
assert HEAD == PINNED_SCIENCE_COMMIT
assert MB == BASELINE
get_ipython().run_line_magic('pip', f'install -q {REPO}')
for name in list(sys.modules):
    if name == 'openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
os.chdir('/content')

In [ ]:
import pytest
from openplaque.rca_source_space_plaque_excess_specificity_v1 import synthetic_self_test
print('Synthetic:', synthetic_self_test())
rc = pytest.main(['-q', str(REPO / 'tests/test_rca_source_space_plaque_excess_specificity_v1.py')])
if rc != 0:
    raise RuntimeError(f'pytest failed with code {rc}')
required = [
    DRIVE_ROOT / 'RCA_Source_Space_Plaque_Quantification_v1/summary.json',
    DRIVE_ROOT / 'RCA_Source_Space_Plaque_Quantification_v1/RCA_source_space_station_quantification.csv',
    DRIVE_ROOT / 'Longitudinal_Plaque_PCAT_Fusion_v1/RCA_source_longitudinal_plaque_profile_1mm.csv',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing prerequisites: ' + repr(missing))
(OUTPUT / 'preflight_complete.json').write_text(json.dumps({'status':'PASS','science_commit':PINNED_SCIENCE_COMMIT,'baseline':BASELINE}, indent=2))
print('Preflight PASS')

In [ ]:
if reuse_complete:
    result = {'summary': json.loads((OUTPUT / 'summary.json').read_text()), 'report': str(OUTPUT / 'OPENPLAQUE_RCA_SOURCE_SPACE_PLAQUE_EXCESS_SPECIFICITY_REPORT.html'), 'zip': str(OUTPUT / 'OPENPLAQUE_RCA_SOURCE_SPACE_PLAQUE_EXCESS_SPECIFICITY_RESULTS.zip')}
    print('Reused completed result')
else:
    from openplaque.rca_source_space_plaque_excess_specificity_v1 import run
    result = run(drive_root=str(DRIVE_ROOT), output_dir=str(OUTPUT))
print(json.dumps(result['summary'], indent=2, default=str))
print('Report:', result['report'])
print('ZIP:', result['zip'])

In [ ]:
from IPython.display import display, Image
for name in ['01_RCA_raw_vs_excess_specificity.png','02_RCA_excess_shell_consistency.png','03_RCA_excess_composition_profile.png']:
    p = OUTPUT / name
    if p.exists():
        display(Image(filename=str(p)))